# 04 - Baselines

Two reference models that learn nothing from our data. Both reduce each scalogram to 16 numbers with PCA, the same size as the VAE latent vector, and then use the same GMM step as the VAE notebooks.

1. Raw PCA: PCA directly on the 3,072 pixels of each scalogram. Answers whether any neural network helps at all.
2. MobileNetV2 PCA: a MobileNetV2 trained on ImageNet photos, with its weights frozen and its classifier removed, gives 1,280 features per scalogram (as in Benavides and Luna, 2024). PCA reduces them to 16.

PCA and the GMM are fitted on the training days only. The last section compares every model that has been run.

<d.benavidess@uniandes.edu.co>

In [2]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import torch

import numpy as np
import pandas as pd

import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split
from src.models.baselines import raw_features, mobilenet_features, pca_reduce
from src.training import set_seed
from src.clustering import assign_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import selection_chart, regime_timeline

## 1. Configuration

In [3]:
SCALOGRAM_DIR = Path("../data/processed/scalograms")
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models/baselines")
FIGURE_DIR = Path("../figures/baselines")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "n_components": 16,
    "image_size": 224,
    "batch_size": 64,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "n_regimes": None,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "seed": 42,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
(MODEL_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(f"Device: {DEVICE}")

Device: cpu


## 2. Data

In [4]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms: {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")

t = PrettyTable(["Split", "Days", "From", "To"])
for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    t.add_row([name, len(idx), context["date"].iloc[idx[0]].date(), context["date"].iloc[idx[-1]].date()])
print(t)

Scalograms: (9709, 1, 128, 24), range [0.000, 1.000]
+-------+------+------------+------------+
| Split | Days |    From    |     To     |
+-------+------+------------+------------+
| train | 5825 | 2000-01-01 | 2015-12-12 |
|  val  | 1942 | 2015-12-13 | 2021-04-06 |
|  test | 1942 | 2021-04-07 | 2026-07-31 |
+-------+------+------------+------------+


In [5]:
def run_baseline(name, latents):
    model_dir = MODEL_DIR / name
    figure_dir = FIGURE_DIR / name
    figure_dir.mkdir(parents=True, exist_ok=True)

    selection, k, labels, probs = assign_regimes(
        latents, train_idx, context["precio_real_cop_kwh"].to_numpy(),
        k=CONFIG["n_regimes"], k_range=CONFIG["k_range"], seed=CONFIG["seed"],
    )
    ctx = context.assign(regime=labels)
    metrics = {"model": name, "k": k, **summary_metrics(latents, labels, seed=CONFIG["seed"])}
    save_regimes(model_dir, ctx, labels, probs, metrics)
    np.save(model_dir / "latents.npy", latents)

    selection_chart(selection, k).save(str(figure_dir / "gmm_selection.png"), scale_factor=2.0)
    timeline = regime_timeline(ctx, split_dates, title=f"{name}: detected regimes (k={k})")
    timeline.save(str(figure_dir / "regime_timeline.png"), scale_factor=2.0)

    t = PrettyTable(["Regime", "Days", "Share %", "Real price", "Reservoir", "Mean ONI", "El Nino %"])
    for _, row in regime_profile(ctx).iterrows():
        t.add_row([int(row["regime"]), int(row["days"]), f"{row['share_pct']:.1f}", f"{row['mean_real_price']:.1f}",
                   f"{row['mean_reservoir']:.3f}", f"{row['mean_oni']:.2f}", f"{row['el_nino_pct']:.1f}"])
    print(f"{name}: k = {k}")
    if CONFIG["n_regimes"] is None and k == max(CONFIG["k_range"]):
        print("BIC is still falling at the largest k. Choose k from the table and set CONFIG['n_regimes'] by hand.")
    print(t)
    return timeline

## 3. Raw PCA

In [6]:
raw_latents, raw_var = pca_reduce(raw_features(images), train_idx, CONFIG["n_components"])
print(f"Explained variance with {CONFIG['n_components']} components: {raw_var.sum():.1%}")

Explained variance with 16 components: 99.1%


In [7]:
raw_timeline = run_baseline("raw_pca", raw_latents)
raw_timeline

c:\Users\Daniel Benavides\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Daniel Benavides\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Daniel Benavides\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Daniel Benavides\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child

raw_pca: k = 8
BIC is still falling at the largest k. Choose k from the table and set CONFIG['n_regimes'] by hand.
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------+------------+-----------+----------+-----------+
|   0    | 1145 |   11.8  |   177.5    |   0.710   |  -0.07   |    18.6   |
|   1    | 1207 |   12.4  |   181.0    |   0.704   |  -0.23   |    13.2   |
|   2    | 1038 |   10.7  |   198.6    |   0.710   |  -0.40   |    9.6    |
|   3    | 1275 |   13.1  |   312.4    |   0.632   |   0.07   |    31.3   |
|   4    | 927  |   9.5   |   332.9    |   0.648   |   0.07   |    31.1   |
|   5    | 925  |   9.5   |   344.1    |   0.692   |   0.22   |    31.9   |
|   6    | 1843 |   19.0  |   383.1    |   0.710   |  -0.01   |    22.1   |
|   7    | 1349 |   13.9  |   526.1    |   0.617   |   0.34   |    40.2   |
+--------+------+---------+------------+---------

alt.VConcatChart(...)

## 4. MobileNetV2 PCA

The first run downloads the ImageNet weights (about 14 MB) and extracts features for all days. The features are cached in `models/baselines/` so later runs skip this step. Each scalogram is repeated to 3 channels, resized from 128 x 24 to 224 x 224 and normalized with ImageNet statistics.

In [8]:
feature_path = MODEL_DIR / "mobilenet_features.npy"
if feature_path.exists():
    mobilenet_feats = np.load(feature_path)
else:
    mobilenet_feats = mobilenet_features(images, DEVICE, CONFIG["batch_size"], CONFIG["image_size"])
    np.save(feature_path, mobilenet_feats)

mobilenet_latents, mobilenet_var = pca_reduce(mobilenet_feats, train_idx, CONFIG["n_components"])
print(f"Features: {mobilenet_feats.shape}")
print(f"Explained variance with {CONFIG['n_components']} components: {mobilenet_var.sum():.1%}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\Daniel Benavides/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:02<00:00, 5.38MB/s]


Features: (9709, 1280)
Explained variance with 16 components: 73.0%


In [9]:
mobilenet_timeline = run_baseline("mobilenet_pca", mobilenet_latents)
mobilenet_timeline

mobilenet_pca: k = 8
BIC is still falling at the largest k. Choose k from the table and set CONFIG['n_regimes'] by hand.
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------+------------+-----------+----------+-----------+
|   0    | 956  |   9.8   |   201.7    |   0.689   |   0.22   |    27.4   |
|   1    | 928  |   9.6   |   232.4    |   0.670   |  -0.20   |    15.6   |
|   2    | 1699 |   17.5  |   251.9    |   0.670   |  -0.15   |    21.3   |
|   3    | 1197 |   12.3  |   259.4    |   0.672   |  -0.20   |    21.0   |
|   4    | 1461 |   15.0  |   271.6    |   0.667   |  -0.05   |    24.3   |
|   5    | 1306 |   13.5  |   272.3    |   0.704   |   0.04   |    23.6   |
|   6    | 1548 |   15.9  |   313.6    |   0.708   |  -0.13   |    19.3   |
|   7    | 614  |   6.3   |   1113.7   |   0.612   |   1.12   |    68.7   |
+--------+------+---------+------------+---

alt.VConcatChart(...)

## 5. Comparison across models

Reads the metrics of every model that has been run. There are no true regime labels, so the comparison uses cluster separation (silhouette, on each model's own latent space), regime persistence (run length, switches per year) and, in each notebook, the regime profile against price, reservoir level and ONI. Silhouette values from different latent spaces are only roughly comparable.

In [10]:
runs = {
    "ConvVAE": Path("../models/vae/metrics.json"),
    "Temporal VAE": Path("../models/temporal_vae/metrics.json"),
    "Raw PCA": MODEL_DIR / "raw_pca" / "metrics.json",
    "MobileNetV2 PCA": MODEL_DIR / "mobilenet_pca" / "metrics.json",
}

t = PrettyTable(["Model", "k", "Silhouette", "Mean run (days)", "Median run (days)", "Switches / year"])
for label, path in runs.items():
    if not path.exists():
        t.add_row([label, "not run", "", "", "", ""])
        continue
    m = json.loads(path.read_text())
    t.add_row([label, m["k"], f"{m['silhouette']:.3f}", f"{m['mean_run_days']:.1f}",
               f"{m['median_run_days']:.0f}", f"{m['switches_per_year']:.1f}"])
print(t)

+-----------------+---+------------+-----------------+-------------------+-----------------+
|      Model      | k | Silhouette | Mean run (days) | Median run (days) | Switches / year |
+-----------------+---+------------+-----------------+-------------------+-----------------+
|     ConvVAE     | 8 |   0.196    |       3.2       |         2         |      115.1      |
|   Temporal VAE  | 7 |   0.377    |       4.7       |         2         |       78.2      |
|     Raw PCA     | 8 |   -0.131   |       1.9       |         1         |      189.4      |
| MobileNetV2 PCA | 8 |   0.139    |       2.8       |         1         |      130.6      |
+-----------------+---+------------+-----------------+-------------------+-----------------+
